# Supplementary Figure 4 | Expression over intergenic null in pooled and tissue cohorts

Fraction of annotated genes in a given biotype expressed at FDR 5% versus intergenic null for novel lncRNA (left), existing lncRNA (middle) and protein-coding genes (right) across GTEx, dGTEx, and Macaque and Marmoset (NHP-dGTEx). Counts indicate the number of genes in each category. Bar color indicates if genes are significantly expressed in a pooled cohort (all samples in the dataset), in a tissue cohort (samples only of that tissue, for tissues with n≥10 samples) or both. For Macaque and Marmoset, the red line marks the fraction of annotated human genes that have an ortholog in that species; expression bars are reported as the fraction of human annotated genes with a called-expressed ortholog. Orthologs are annotated for lncRNAs specifically, so protein-coding expression in Macaque and Marmoset is not included.


## Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from tqdm import tqdm

plt.rc("axes", labelsize=6, linewidth=0.6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6, family="sans-serif")
plt.rc("axes", titlesize=7)
plt.rc("figure", titlesize=7)
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["xtick.major.width"] = 0.6
plt.rcParams["ytick.major.width"] = 0.6
plt.rcParams["xtick.major.size"] = 2.5
plt.rcParams["ytick.major.size"] = 2.5

fig_width, fig_height = 7.2, 2.7
min_n = 10  # min samples for tissue-detail call
min_pct_inside = 5.5  # below this, count label goes outside with a leader line
ortholog_line_color = "#e31a1c"

# GTEx tissue cohorts: all sites with n>=min_n except histology subtypes (54 sites = 52 tau tissues + 2 cell lines)
histology_subtypes = {
    "Colon - Transverse - Muscularis",
    "Stomach - Mixed Cell",
    "Pancreas - Mixed Cell",
    "Small Intestine - Terminal Ileum - Mixed Cell",
    "Stomach - Mucosa",
    "Stomach - Muscularis",
    "Pancreas - Acini",
    "Small Intestine - Terminal Ileum - Lymphoid Aggregate",
    "Colon - Transverse - Mixed Cell",
    "Colon - Transverse - Mucosa",
    "Liver - Mixed Cell",
    "Pancreas - Islets",
    "Liver - Hepatocyte",
    "Liver - Portal Tract",
}

gene_class_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding"]
gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
}
# Protein-coding: human cohorts only (no NHP ortholog mapping)
gene_class_datasets = {
    "novel_lncRNA": ["GTEx", "dGTEx", "Macaque", "Marmoset"],
    "existing_lncRNA": ["GTEx", "dGTEx", "Macaque", "Marmoset"],
    "protein_coding": ["GTEx", "dGTEx"],
}
cat_order = ["both", "pooled_only", "tissue_only"]
cat_pretty = {
    "both": "pooled & tissue",
    "pooled_only": "pooled only",
    "tissue_only": "tissue only",
}
cat_colors = {
    "both": "#252525",
    "pooled_only": "#969696",
    "tissue_only": "#f7f7f7",
}
datasets = ["GTEx", "dGTEx", "Macaque", "Marmoset"]


## Paths

In [ ]:
gencode_class_file = Path("path/to/gencode_v39_v47_gene_classification.txt")
lnc_category_file = Path("path/to/lncrna_category.txt")
sample_attributes_file = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")
ortholog_file = Path("path/to/dGTEx_nhpGTEx_liftedlncRNAs.tsv")

gtex_null_file = Path("path/to/GTEx.null_quantifications.txt")
gtex_by_tissue_null_dir = Path("path/to/by_tissue/GTEx")
dgtex_null_file = Path("path/to/dGTEx.null_quantifications.txt")
macaque_null_file = Path("path/to/macaque.null_quantifications.txt")
marmoset_null_file = Path("path/to/marmoset.null_quantifications.txt")
dgtex_by_tissue_dir = Path("path/to/by_tissue/dGTEx")
macaque_by_tissue_dir = Path("path/to/by_tissue/macaque")
marmoset_by_tissue_dir = Path("path/to/by_tissue/marmoset")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_4_pooled_vs_tissue_expression.pdf"
fig_png = fig_dir / "supp_figure_4_pooled_vs_tissue_expression.png"


## Load data


### lncRNA category


In [ ]:
lnc = pd.read_table(lnc_category_file, usecols=["gene_id", "gene_id_stable"])
print(f"lncRNA category rows: {len(lnc):,}")


### GENCODE classification


In [ ]:
gencode = pd.read_table(
    gencode_class_file,
    usecols=["gene_id_stable", "gene_id_v47", "gene_type_grouped", "in_v47_full"],
    low_memory=False,
)
print(f"GENCODE rows: {len(gencode):,}")


### Gene class sets


In [ ]:
lnc = lnc.merge(gencode, on="gene_id_stable", how="left")
lnc = lnc.loc[lnc["in_v47_full"].fillna(False)].copy()
novel = set(lnc.loc[lnc["gene_type_grouped"] == "novel_lncRNA", "gene_id"].astype(str))
existing = set(lnc.loc[lnc["gene_type_grouped"] == "existing_lncRNA", "gene_id"].astype(str))

pc = gencode.loc[
    gencode["in_v47_full"].fillna(False) & (gencode["gene_type_grouped"] == "protein_coding")
].copy()
protein_coding = set(pc["gene_id_v47"].astype(str))

gene_sets = {
    "novel_lncRNA": novel,
    "existing_lncRNA": existing,
    "protein_coding": protein_coding,
}
print(
    f"v47 novel={len(novel):,}; existing={len(existing):,}; "
    f"protein-coding={len(protein_coding):,}"
)


### NHP ortholog map


In [ ]:
orth = pd.read_csv(ortholog_file, sep="\t")
for col in ["gene_id.macaque", "gene_id.marmoset"]:
    orth[col] = orth[col].replace(["NA", "-"], pd.NA)
orth_by = orth.drop_duplicates("gene_id.human").set_index("gene_id.human")
mac_map = orth_by["gene_id.macaque"].to_dict()
mar_map = orth_by["gene_id.marmoset"].to_dict()
print(f"ortholog rows: {len(orth_by):,}")


### GTEx pooled null


In [ ]:
gtex_null = pd.read_table(gtex_null_file)
print(f"GTEx pooled null: {len(gtex_null):,}")


### dGTEx pooled null


In [ ]:
dgtex_null = pd.read_csv(dgtex_null_file)
print(f"dGTEx pooled null: {len(dgtex_null):,}")


### Macaque pooled null


In [ ]:
macaque_null = pd.read_csv(macaque_null_file)
print(f"macaque pooled null: {len(macaque_null):,}")


### Marmoset pooled null


In [ ]:
marmoset_null = pd.read_csv(marmoset_null_file)
print(f"marmoset pooled null: {len(marmoset_null):,}")


### GTEx sample metadata


In [ ]:
gtex_sample_meta = pd.read_csv(sample_attributes_file, sep="\t", low_memory=False)
print(f"GTEx sample meta rows: {len(gtex_sample_meta):,}")


## Run analysis


### Pooled vs tissue-detail call sets


In [ ]:
def _tissue_file_stem(tissue_id: str) -> str:
    return tissue_id.replace("(", "").replace(")", "").replace(" - ", "_").replace(" ", "_")


def gtex_pooled_tissue(min_n: int = min_n):
    pooled = set(gtex_null.loc[gtex_null["GTEx.fdr05_quantified"], "gene_id"].astype(str))
    n_by = gtex_sample_meta.loc[gtex_sample_meta["SMAFRZE"] == "RNASEQ"].groupby("SMTSD").size()
    any_t = set()
    used = []
    for smtsd, n in tqdm(list(n_by.items()), desc="GTEx tissues"):
        if n < min_n or smtsd in histology_subtypes:
            continue
        path = gtex_by_tissue_null_dir / f"{_tissue_file_stem(smtsd)}.null_quantifications.txt"
        df = pd.read_table(path)
        fcol = [c for c in df.columns if c.endswith(".fdr05_quantified")][0]
        any_t |= set(df.loc[df[fcol], "gene_id"].astype(str))
        used.append(smtsd)
    print(
        f"GTEx tissue-detail call set: {len(used)} sites "
        f"(n≥{min_n}; 52 τ + cell lines; subtypes excluded)"
    )
    return pooled, any_t


def cohort_pooled_tissue(pooled_df: pd.DataFrame, tissue_dir: Path, gene_col: str = "Gene"):
    pooled = set(pooled_df.loc[pooled_df["fdr05_quantified"], gene_col].astype(str))
    paths = sorted(tissue_dir.glob("*.wilcoxon.csv"))
    if not paths:
        raise FileNotFoundError(f"No *.wilcoxon.csv in {tissue_dir}")
    any_t = set()
    for path in tqdm(paths, desc=tissue_dir.name):
        df = pd.read_csv(path)
        n_samp = int(df["n_samples"].iloc[0]) if "n_samples" in df.columns else min_n
        if n_samp < min_n:
            continue
        any_t |= set(df.loc[df["fdr05_quantified"], gene_col].astype(str))
    return pooled, any_t


def species_to_human(ids: set[str], id_map: dict) -> set[str]:
    inv = {}
    for human, sp in id_map.items():
        if pd.notna(sp):
            inv.setdefault(str(sp), set()).add(str(human))
    out = set()
    for i in ids:
        out |= inv.get(i, set())
    return out


parts = {}
parts["GTEx"] = gtex_pooled_tissue()
parts["dGTEx"] = cohort_pooled_tissue(dgtex_null, dgtex_by_tissue_dir)
mac_p, mac_t = cohort_pooled_tissue(macaque_null, macaque_by_tissue_dir)
parts["Macaque"] = (species_to_human(mac_p, mac_map), species_to_human(mac_t, mac_map))
mar_p, mar_t = cohort_pooled_tissue(marmoset_null, marmoset_by_tissue_dir)
parts["Marmoset"] = (species_to_human(mar_p, mar_map), species_to_human(mar_t, mar_map))

rows = []
for gene_class, gene_set in gene_sets.items():
    for dataset in gene_class_datasets[gene_class]:
        pooled, tissue = parts[dataset]
        both = (pooled & tissue) & gene_set
        pooled_only = (pooled - tissue) & gene_set
        tissue_only = (tissue - pooled) & gene_set
        for cat, ids in [("both", both), ("pooled_only", pooled_only), ("tissue_only", tissue_only)]:
            rows.append(
                {
                    "dataset": dataset,
                    "gene_class": gene_class,
                    "category": cat,
                    "n": len(ids),
                }
            )
breakdown = pd.DataFrame(rows)

# Human genes in each class with a non-null NHP ortholog (ceiling for NHP bars)
nhp_orth_maps = {"Macaque": mac_map, "Marmoset": mar_map}
ortholog_n = {}
for gene_class, gene_set in gene_sets.items():
    for dataset, id_map in nhp_orth_maps.items():
        if dataset not in gene_class_datasets[gene_class]:
            continue
        have = {str(h) for h, sp in id_map.items() if pd.notna(sp)}
        ortholog_n[(gene_class, dataset)] = len(gene_set & have)

print(breakdown.groupby(["gene_class", "dataset"])["n"].sum().to_string())


## Make figure


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(fig_width, fig_height), sharey=True)
x = np.arange(len(datasets))
width = 0.52

legend_handles = [
    Patch(
        facecolor=cat_colors[cat],
        edgecolor="#595959",
        linewidth=0.4,
        label=cat_pretty[cat],
    )
    for cat in cat_order
]


def _label_outside(ax, x0, y_seg, count, stack_i, n_stack):
    """Place count to the right of a thin segment with a short leader line."""
    x_right = x0 + width / 2
    dy = 0.0 if n_stack == 1 else (stack_i - (n_stack - 1) / 2) * 2.8
    y_text = y_seg + dy
    x_text = x_right + 0.22
    ax.plot(
        [x_right, x_right + 0.08, x_text - 0.02],
        [y_seg, y_seg, y_text],
        color="#333333",
        linewidth=0.4,
        solid_capstyle="round",
        clip_on=False,
        zorder=5,
    )
    ax.text(
        x_text,
        y_text,
        f"{count:,}",
        ha="left",
        va="center",
        fontsize=4.5,
        color="black",
        clip_on=False,
        zorder=6,
    )


for ax, gene_class in zip(axes, gene_class_order):
    sub = breakdown[breakdown["gene_class"] == gene_class]
    ds_shown = gene_class_datasets[gene_class]
    n_annot = len(gene_sets[gene_class])
    bottom = np.zeros(len(datasets))
    seg_info = {i: [] for i in range(len(datasets))}

    for cat in cat_order:
        counts = []
        for ds in datasets:
            if ds not in ds_shown:
                counts.append(0)
                continue
            hit = sub.loc[(sub["dataset"] == ds) & (sub["category"] == cat), "n"]
            counts.append(int(hit.iloc[0]) if len(hit) else 0)
        pcts = 100.0 * np.asarray(counts, dtype=float) / n_annot
        ax.bar(
            x,
            pcts,
            width,
            bottom=bottom,
            color=cat_colors[cat],
            edgecolor="#595959",
            linewidth=0.35,
            clip_on=True,
        )
        for i, (count, pct) in enumerate(zip(counts, pcts)):
            if count > 0:
                seg_info[i].append(
                    {
                        "cat": cat,
                        "count": count,
                        "pct": pct,
                        "y_mid": bottom[i] + pct / 2,
                    }
                )
        bottom += pcts

    for i, segs in seg_info.items():
        outside = [s for s in segs if s["pct"] < min_pct_inside]
        for s in segs:
            if s["pct"] >= min_pct_inside:
                txt_color = "white" if s["cat"] == "both" else "black"
                ax.text(
                    x[i],
                    s["y_mid"],
                    f"{s['count']:,}",
                    ha="center",
                    va="center",
                    fontsize=4.5,
                    color=txt_color,
                    clip_on=True,
                )
        for j, s in enumerate(outside):
            _label_outside(ax, x[i], s["y_mid"], s["count"], j, len(outside))

    # Ortholog ceiling: fraction of annotated human genes with an NHP ortholog
    for i, ds in enumerate(datasets):
        key = (gene_class, ds)
        if key not in ortholog_n:
            continue
        n_orth = ortholog_n[key]
        pct_orth = 100.0 * n_orth / n_annot
        ax.hlines(
            pct_orth,
            x[i] - width / 2,
            x[i] + width / 2,
            colors=ortholog_line_color,
            linewidths=1.0,
            zorder=4,
            clip_on=False,
        )
        ax.text(
            x[i] + width / 2 + 0.06,
            pct_orth,
            f"{n_orth:,}",
            ha="left",
            va="center",
            fontsize=4.5,
            color=ortholog_line_color,
            clip_on=False,
            zorder=6,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=0)
    if gene_class == "protein_coding":
        for tick_label, ds in zip(ax.get_xticklabels(), datasets):
            if ds not in ds_shown:
                tick_label.set_color("#b0b0b0")
    ax.set_xlabel(gene_type_pretty[gene_class], color="black", labelpad=3)
    ax.tick_params(axis="both", which="major", pad=1.5)
    ax.set_xlim(-0.55, len(datasets) - 0.25)
    sns.despine(ax=ax)
    ax.legend(
        handles=legend_handles,
        frameon=False,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=3,
        fontsize=5,
        handlelength=0.9,
        handleheight=0.9,
        handletextpad=0.35,
        columnspacing=0.9,
        borderaxespad=0.0,
    )

axes[0].set_ylabel("% of annotated genes expressed (FDR 5%)")
axes[0].set_ylim(0, 100)
axes[0].yaxis.set_major_locator(MultipleLocator(20))

fig.tight_layout(w_pad=1.35, rect=(0, 0, 1, 0.96))
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
# No numbers unique to this figure are cited in the paper; Fig. 1 owns the detection counts.
